# 02_ex2_mapping_and_views.sql  Exercise 2：共通機種IDで統合する（20分）

方針 : 元システムのデータは一切変更しない。対応関係は Databricks 側の Delta 表で管理する。  
確定的な集計に使うのは approval_status = 'APPROVED' の対応だけ。  

> **💡 解説**
> - **なぜ**：対応関係を Databricks 側の表で管理し、元システムを変えずに、計画・販売・生産をつなぎます。
> - **仕組み**：確定対応（APPROVED）だけを使う View を作り、実績や計画の各行に共通機種IDを付けます。View なので、対応表を更新すれば結果も自動で変わります。

In [ ]:
%run ./00_config

このノートブックの SQL は `run_sql()`（`00_config` で定義）で実行します。テーブル・View・関数の名前には、`00_config` のカタログ・スキーマが自動で付きます（例：`sales_actual` → `workspace.vehicle_alias_handson.sales_actual`）。**最初に `%run ./00_config` のセルを実行してください。**

## 2-0. 過去の人手承認履歴（Excel）を Volume に置いて取り込む

各部署が過去に人手で作った対応表（Excel）を、Unity Catalog の **Volume** に置き、Delta テーブル `alias_mapping_history` として取り込みます。後の 2-3・2-4 で、この履歴を名称の照合と対応表への取り込みに使います。

> **💡 解説**
> - **なぜ**：属人的な Excel は、共有フォルダやメールに散らばり、「どれが最新か」「誰が見てよいか」が分からなくなりがちです。まず Unity Catalog の管理下に置くことで、置き場所・権限・履歴を1か所にそろえます。
> - **仕組み**：Volume は、テーブルにする前のファイル（Excel・CSV・PDF など）を置くための、Unity Catalog の保存場所です。`/Volumes/<カタログ>/<スキーマ>/<Volume名>/` というパスでアクセスでき、テーブルと同じように `GRANT` で権限を付けられます。
> - **利点**：元の Excel をそのまま残したまま、取り込んだテーブルに「どのファイルから・いつ取り込んだか」を記録できます。元ファイルと取り込み結果の対応を後から確認できるので、監査や問い合わせに答えやすくなります。

### 2-0-1. 取り込み用の Volume を作る

> **💡 解説**
> - **仕組み**：`CREATE VOLUME` は、`00_config` のスキーマの下に Volume `handson_files` を作ります（`run_sql()` が名前にカタログ・スキーマを付けます）。ストレージの場所を指定しない **マネージド Volume** なので、保存先のクラウドストレージは Unity Catalog が管理します。
> - **利点**：ストレージのパスや認証情報を参加者が意識する必要がありません。Free Edition でもそのまま作れます。

In [ ]:
run_sql("""
CREATE VOLUME IF NOT EXISTS handson_files
COMMENT 'ハンズオン用の元ファイル置き場（人手で作成した対応履歴の Excel など・架空データ）'
""")
VOLUME_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/handson_files"
EXCEL_NAME = "alias_mapping_history.xlsx"
EXCEL_PATH = f"{VOLUME_DIR}/{EXCEL_NAME}"
print("Volume:", VOLUME_DIR)

### 2-0-2. Excel を Volume に置く

**画面から置く場合（体験としておすすめ）：**
1. GitHub のリポジトリから `data/alias_mapping_history.xlsx` をダウンロードします（Git フォルダで取り込んだ場合は、ワークスペースの `data/` フォルダにあります）。
2. 左のサイドバーの **カタログ** → `00_config` のカタログ → スキーマ → **ボリューム** → `handson_files` を開きます。
3. **このボリュームにアップロード** をクリックし、Excel を選んでアップロードします。

下のセルは、Volume に Excel が無い場合に、ノートブックの近く（Git フォルダの `data/` など）にある Excel を自動でコピーします。どちらにも無い場合は、アップロード方法を表示して止まります。

> **💡 解説**
> - **仕組み**：サーバーレスのノートブックからは、Volume が `/Volumes/...` という普通のファイルパスとして見えます。そのため、Python の `shutil` や `open()` でそのまま読み書きできます。ワークスペースのファイル（Git フォルダ内の `data/` など）も `/Workspace/...` で読めます。
> - **利点**：Volume に置いた後は、ノートブック・SQL（`read_files`）・ジョブ・パイプラインのどれからでも、同じパスで同じファイルを読めます。

In [ ]:
import os
import posixpath
import shutil

if os.path.exists(EXCEL_PATH):
    print("✅ Volume に Excel があります:", EXCEL_PATH)
else:
    notebook_dir = "/Workspace" + posixpath.dirname(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    candidates = [posixpath.normpath(posixpath.join(notebook_dir, rel, EXCEL_NAME)) for rel in ("../data", "data", ".")]
    found = next((c for c in candidates if os.path.exists(c)), None)
    if found is None:
        raise FileNotFoundError(
            f"Volume に Excel がありません。{EXCEL_NAME} を {VOLUME_DIR} にアップロードしてから、このセルを再実行してください。\n"
            f"（Excel は GitHub の data/{EXCEL_NAME} にあります。探した場所: {candidates}）")
    shutil.copyfile(found, EXCEL_PATH)
    print(f"✅ {found} を Volume にコピーしました:", EXCEL_PATH)

display(dbutils.fs.ls(VOLUME_DIR))

### 2-0-3. Excel を読み、列名と型をそろえる

Excel のシート「対応履歴」は、1〜2行目が表題と注記、**3行目が見出し**、4行目以降がデータです。日本語の見出しをテーブルの列名（英語）に対応づけ、空欄は NULL、日付は日付型にそろえます。

> **💡 解説**
> - **なぜ**：人が作った Excel は、表題行・結合セル・空欄・日付の書式などが混ざっていて、そのままではテーブルになりません。取り込みの時点で「見出しは何行目か」「どの列をどの列名にするか」を**コードで明示**しておけば、毎回同じ結果になります。
> - **仕組み**：`openpyxl` で Excel を読み込みます（サーバーレスに入っていない場合は、このセルでインストールします）。`data_only=True` を指定すると、数式ではなく計算済みの値を読みます。見出しの対応表 `COLUMN_MAP` に無い列は無視し、必要な見出しが欠けていればエラーで止めます。
> - **利点**：Excel の列の並び替えや、余計な列の追加があっても取り込みが壊れにくくなります。必要な列が消えた場合は、黙って誤ったデータを作らずにエラーで気づけます。
> - **補足**：この程度の件数（数十〜数千行）なら、この方法で十分です。大量のファイルを定期的に取り込む場合は、Auto Loader や Lakeflow のパイプラインで、Volume に置かれたファイルを自動で取り込む形にします。

In [ ]:
import datetime as dt

try:
    import openpyxl
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])
    import openpyxl

SHEET_NAME = "対応履歴"
HEADER_ROW = 3   # 見出しの行（1行目が表題、2行目が注記）
COLUMN_MAP = {   # Excel の見出し → テーブルの列名
    "履歴ID": "history_id", "元システム": "source_system", "国": "country", "業務領域": "business_area",
    "元の名称": "source_name", "元のコード": "source_code", "対応づけた共通機種ID": "mapped_vehicle_id",
    "適用開始日": "valid_from", "適用終了日": "valid_to", "状態": "history_status",
    "承認部署": "approved_by", "承認日": "approved_at", "元資料": "source_document",
}
DATE_COLUMNS = {"valid_from", "valid_to", "approved_at"}


def to_date(value):
    if value is None or (isinstance(value, str) and not value.strip()):
        return None
    if isinstance(value, dt.datetime):
        return value.date()
    if isinstance(value, dt.date):
        return value
    return dt.datetime.strptime(str(value).strip().replace("/", "-"), "%Y-%m-%d").date()


def to_text(value):
    if value is None:
        return None
    text = str(value).strip()
    return text or None


workbook = openpyxl.load_workbook(EXCEL_PATH, read_only=True, data_only=True)
rows = list(workbook[SHEET_NAME].iter_rows(values_only=True))
header = [to_text(h) for h in rows[HEADER_ROW - 1]]
missing = [h for h in COLUMN_MAP if h not in header]
if missing:
    raise ValueError(f"Excel に必要な見出しがありません: {missing}（見出しは {HEADER_ROW} 行目の想定）")

records = []
for row in rows[HEADER_ROW:]:
    if all(v is None for v in row):
        continue   # 空行は読み飛ばす
    values = {COLUMN_MAP[h]: v for h, v in zip(header, row) if h in COLUMN_MAP}
    records.append({col: (to_date(v) if col in DATE_COLUMNS else to_text(v)) for col, v in values.items()})

print(f"✅ {len(records)} 行を読み込みました（シート「{SHEET_NAME}」、見出しは {HEADER_ROW} 行目）")
for r in records:
    print(r["history_id"], r["source_name"], r["source_code"], r["mapped_vehicle_id"], r["history_status"])

### 2-0-4. Delta テーブルとして保存する

読み込んだ行を Delta テーブル `alias_mapping_history` に保存します。取り込み元のファイルと取り込み日時も列として残します。

> **💡 解説**
> - **なぜ**：Excel のままでは、SQL で他のテーブルと結合したり、Genie から参照したりできません。Delta テーブルにすると、対応表・実績データと同じように扱えます。
> - **仕組み**：列の型をスキーマ（`StructType`）で明示してから DataFrame を作り、`saveAsTable` で保存します。`mode("overwrite")` なので、Excel を差し替えて再実行すると、テーブルも最新の内容に置き換わります。`source_file`・`ingested_at` 列に取り込み元と日時を残します。
> - **利点**：Delta はテーブルの変更履歴を持つので、「いつの Excel を取り込んだ結果か」を `DESCRIBE HISTORY` やタイムトラベル（`VERSION AS OF`）で後から確認できます。元の Excel も Volume に残っているので、取り込み結果と元ファイルを突き合わせられます。

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

HISTORY_SCHEMA = T.StructType([
    T.StructField("history_id", T.StringType(), False),
    T.StructField("source_system", T.StringType()),
    T.StructField("country", T.StringType()),
    T.StructField("business_area", T.StringType()),
    T.StructField("source_name", T.StringType()),
    T.StructField("source_code", T.StringType()),
    T.StructField("mapped_vehicle_id", T.StringType()),
    T.StructField("valid_from", T.DateType()),
    T.StructField("valid_to", T.DateType()),
    T.StructField("history_status", T.StringType()),
    T.StructField("approved_by", T.StringType()),
    T.StructField("approved_at", T.DateType()),
    T.StructField("source_document", T.StringType()),
])

history_df = (spark.createDataFrame([tuple(r[f.name] for f in HISTORY_SCHEMA.fields) for r in records], HISTORY_SCHEMA)
              .withColumn("source_file", F.lit(EXCEL_PATH))
              .withColumn("ingested_at", F.current_timestamp()))
(history_df.write.mode("overwrite").option("overwriteSchema", "true")
 .saveAsTable(f"{CATALOG}.{SCHEMA}.alias_mapping_history"))

run_sql("""
COMMENT ON TABLE alias_mapping_history IS '過去の人手対応履歴（架空）。Volume の Excel（source_file）から取り込んだもの。一部のみ存在し、名称だけのものもある。確定マスタではない。';
ALTER TABLE alias_mapping_history SET TAGS ('handson_business_domain' = 'Vehicle Profitability', 'data_role' = 'mapping_history');
SELECT history_id, source_name, source_code, mapped_vehicle_id, valid_from, valid_to, history_status, source_file, ingested_at
FROM alias_mapping_history
ORDER BY history_id
""")

## 2-1. 正規化関数の動作確認

> **💡 解説**
> - **なぜ**：正規化関数が表記揺れをどうそろえるか、どこから先はそろえないか（簡体字と繁体字）を先に確かめます。
> - **仕組み**：`VALUES` 句でその場限りの表を作り、関数を試します。

In [ ]:
run_sql(r"""
SELECT raw, norm_code(raw) AS code_normalized, norm_name(raw) AS name_normalized
FROM VALUES ('JP-A11'), ('jp-a11 '), ('ＪＰ－Ａ１１'), ('MTO-987'), ('mto-987 '),
            ('ALPHA 11 M'), ('Ａｌｐｈａ　１１Ｍ'), ('阿尔法'), ('阿爾法') AS t(raw);
""")

## 2-2. 【Before】確定マスタの「完全一致」だけで変換した場合のカバー率

> **💡 解説**
> - **なぜ**：完全一致だけで、どこまで変換できるか（Before）を数字で押さえます。
> - **仕組み**：`LEFT JOIN` で、対応表に一致しなかった行も残し、一致した行の割合を計算します。適用期間（`BETWEEN valid_from AND valid_to`）も条件に入れています。
> - **利点**：後でルールを足したときの改善を、同じ指標で比べられます。

In [ ]:
run_sql(r"""
SELECT
  count(*)                                         AS sales_rows,
  count(a.alias_id)                                AS matched_rows,
  round(count(a.alias_id) / count(*) * 100, 1)     AS match_rate_pct
FROM sales_actual s
LEFT JOIN vehicle_alias_master a
  ON a.approval_status = 'APPROVED'
 AND a.business_area   = 'SALES'
 AND a.local_vehicle_code = s.sales_vehicle_code
 AND s.sales_month BETWEEN a.valid_from AND a.valid_to;
-- 期待値（2-4 の MERGE 実行前）: 26 行中 18 行（69.2%）。
--   落ちる 8 行 = 表記揺れ 2行（S006, S009）・旧中国コード 2行（S013, S014）・候補のみ/未登録 4行（S023〜S026）
--   ※ 2-4 実行後に再実行すると S013, S014 が一致し 20 行（76.9%）になる
""")

## 2-3. 名称の表記揺れを正規化して、名称のみの過去履歴を確認する

→ 候補が 1つに絞れるものと、複数候補（要確認）になるものがある  

> **💡 解説**
> - **なぜ**：コードが無く名称だけの履歴は、名称を正規化して照合するしかありません。ただし、候補が複数ある場合は自動で決めずに要確認にします。
> - **仕組み**：`collect_set` で候補の共通機種IDを重複なしで集め、`size` で数を数えて判定します。
> - **利点**：名称だけの情報でも、使える部分（一意に決まるもの）と、人が見るべき部分を分けられます。

In [ ]:
run_sql(r"""
SELECT
  h.history_id,
  h.source_name,
  norm_name(h.source_name)                          AS normalized_name,
  array_sort(collect_set(a.canonical_vehicle_id))   AS candidate_vehicle_ids,
  CASE size(collect_set(a.canonical_vehicle_id))
    WHEN 0 THEN '未解決（名称が一致しない。簡体字/繁体字など辞書が必要）'
    WHEN 1 THEN '一意に特定（ただしコード・期間で最終確認）'
    ELSE        '要確認（同名の別世代・別機種が存在）'
  END                                               AS judgement
FROM alias_mapping_history h
LEFT JOIN vehicle_alias_master a
  ON a.approval_status = 'APPROVED'
 AND norm_name(a.local_vehicle_name) = norm_name(h.source_name)
WHERE h.source_code IS NULL
GROUP BY h.history_id, h.source_name
ORDER BY h.history_id;
-- 期待値: H002/H003 → [VEHICLE-001]、H004 'alpha' → [VEHICLE-001, VEHICLE-002]（要確認）、H005 '阿爾法' → 未解決
""")

## 2-4. 過去の人手承認履歴（コードあり・APPROVED）を対応表に取り込む

元の Excel は変更しない。取込元・承認者・根拠資料を残す。  

> **💡 解説**
> - **なぜ**：人手で承認済みで、コードと適用期間がある履歴だけを、確定対応として対応表に取り込みます。元の Excel は変更しません。
> - **仕組み**：`MERGE INTO` は、キー（alias_id）が一致すれば更新し、無ければ追加する命令です。何度実行しても結果が同じ（べき等）になります。
> - **利点**：`mapping_method` を HISTORY にし、`source_document` と `approved_by` に元資料と承認部署を残すので、なぜ確定としたかを後から説明できます。

In [ ]:
run_sql(r"""
MERGE INTO vehicle_alias_master AS t
USING (
  SELECT
    concat('HIST-', history_id)  AS alias_id,
    mapped_vehicle_id            AS canonical_vehicle_id,
    country,
    business_area,
    source_name                  AS local_vehicle_name,
    source_code                  AS local_vehicle_code,
    valid_from,
    valid_to,
    'APPROVED'                   AS approval_status,
    97                           AS confidence_score,
    'HISTORY'                    AS mapping_method,
    source_document,
    approved_by,
    concat('過去の人手対応履歴から取込（承認日 ', CAST(approved_at AS STRING), '）') AS remarks,
    current_timestamp()          AS updated_at
  FROM alias_mapping_history
  WHERE history_status = 'APPROVED'
    AND source_code IS NOT NULL
    AND mapped_vehicle_id IS NOT NULL
    AND valid_from IS NOT NULL AND valid_to IS NOT NULL
) AS s
ON t.alias_id = s.alias_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
""")

> **💡 解説**
> - **なぜ**：取り込まれた行を確認します。

In [ ]:
run_sql(r"""
SELECT * FROM vehicle_alias_master WHERE mapping_method = 'HISTORY';
-- 期待値: HIST-H001（CN-X123B → VEHICLE-001、2024-07-01〜2025-05-31）
""")

## 2-5. 確定済み対応（APPROVED かつ必須項目あり）の View

> **💡 解説**
> - **なぜ**：集計に使ってよい対応を「APPROVED かつ必須項目あり」に限る、ただ1つの入り口を作ります。
> - **仕組み**：View は SQL の定義だけを保存し、参照するたびに元のテーブルから計算します。正規化済みのコードもここで計算しておきます。
> - **利点**：候補や登録途中の行が、うっかり集計に混ざることを防げます。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_alias_approved
COMMENT '確定済みの名称・コード対応。集計に使用してよいのはこの View の対応のみ。'
AS
SELECT
  alias_id, canonical_vehicle_id, country, business_area,
  local_vehicle_name, local_vehicle_code,
  norm_code(local_vehicle_code) AS norm_vehicle_code,
  valid_from, valid_to, confidence_score, mapping_method, source_document
FROM vehicle_alias_master
WHERE approval_status = 'APPROVED'
  AND canonical_vehicle_id IS NOT NULL
  AND local_vehicle_code   IS NOT NULL
  AND valid_from IS NOT NULL
  AND valid_to   IS NOT NULL;
""")

## 2-6. 各実績・計画レコードを共通機種IDへ変換する View

ルール:  
(1) 業務領域（SALES / PRODUCTION / DEVELOPMENT）が一致すること（同じ文字列でも別システムなら別物）  
(2) 正規化後のコードが一致すること  
(3) レコードの月が適用期間内であること  
(4) 候補が 1つの共通機種IDに絞れること。2つ以上なら CONFLICT として集計から除外（誤結合より要確認を優先）  

> **💡 解説**
> - **なぜ**：販売実績の各行に共通機種IDを付けます。4つの条件（業務領域・正規化したコード・適用期間・一意性）をすべて満たす場合だけ変換します。
> - **仕組み**：`LEFT JOIN` で候補をすべてつないだ後、行ごとに `GROUP BY` します。共通機種IDが1つなら RESOLVED、0なら UNRESOLVED、2つ以上なら CONFLICT とします。
> - **利点**：別の車両の数字が誤って混ざるより、変換せずに要確認として残す設計になります。どのルールで変換したか（`resolution_method`）も残ります。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_sales_resolved
COMMENT '販売実績に共通機種IDを付与した View（RESOLVED / UNRESOLVED / CONFLICT）'
AS
WITH matched AS (
  SELECT s.*, a.canonical_vehicle_id, a.alias_id,
         CASE WHEN a.alias_id IS NULL THEN NULL
              WHEN s.sales_vehicle_code = a.local_vehicle_code THEN concat('EXACT:', a.mapping_method)
              ELSE concat('NORMALIZED:', a.mapping_method) END AS method
  FROM sales_actual s
  LEFT JOIN v_alias_approved a
    ON a.business_area     = 'SALES'
   AND a.country           = s.country
   AND a.norm_vehicle_code = norm_code(s.sales_vehicle_code)
   AND s.sales_month BETWEEN a.valid_from AND a.valid_to
)
SELECT
  sales_record_id, country, sales_vehicle_code, powertrain, body_type,
  sales_month, sales_volume, actual_sales, source_system,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 1 THEN max(canonical_vehicle_id) END AS canonical_vehicle_id,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 0 THEN 'UNRESOLVED'
       WHEN count(DISTINCT canonical_vehicle_id) > 1 THEN 'CONFLICT'
       ELSE 'RESOLVED' END                                   AS resolution_status,
  nullif(concat_ws(',', array_sort(collect_set(method))), '')   AS resolution_method,
  nullif(concat_ws(',', array_sort(collect_set(alias_id))), '') AS alias_ids
FROM matched
GROUP BY sales_record_id, country, sales_vehicle_code, powertrain, body_type,
         sales_month, sales_volume, actual_sales, source_system;
""")

> **💡 解説**
> - **なぜ**：生産実績も、販売と同じ4つの条件で変換します。業務領域（PRODUCTION）で分けるので、同じ文字列のコードがあっても販売とは混ざりません。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_production_resolved
COMMENT '生産実績に共通機種IDを付与した View（RESOLVED / UNRESOLVED / CONFLICT）'
AS
WITH matched AS (
  SELECT p.*, a.canonical_vehicle_id, a.alias_id,
         CASE WHEN a.alias_id IS NULL THEN NULL
              WHEN p.mto_code = a.local_vehicle_code THEN concat('EXACT:', a.mapping_method)
              ELSE concat('NORMALIZED:', a.mapping_method) END AS method
  FROM production_actual p
  LEFT JOIN v_alias_approved a
    ON a.business_area     = 'PRODUCTION'
   AND a.norm_vehicle_code = norm_code(p.mto_code)
   AND p.production_month BETWEEN a.valid_from AND a.valid_to
)
SELECT
  production_record_id, plant_code, mto_code, production_month,
  production_volume, actual_material_cost,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 1 THEN max(canonical_vehicle_id) END AS canonical_vehicle_id,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 0 THEN 'UNRESOLVED'
       WHEN count(DISTINCT canonical_vehicle_id) > 1 THEN 'CONFLICT'
       ELSE 'RESOLVED' END                                   AS resolution_status,
  nullif(concat_ws(',', array_sort(collect_set(method))), '')   AS resolution_method,
  nullif(concat_ws(',', array_sort(collect_set(alias_id))), '') AS alias_ids
FROM matched
GROUP BY production_record_id, plant_code, mto_code, production_month,
         production_volume, actual_material_cost;
""")

> **💡 解説**
> - **なぜ**：開発計画にも、同じ仕組みで共通機種IDを付けます。3つの系統が同じ ID を持つことで、初めて横断して集計できます。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_plan_resolved
COMMENT '開発計画に共通機種IDを付与した View（RESOLVED / UNRESOLVED / CONFLICT）'
AS
WITH matched AS (
  SELECT d.*, a.canonical_vehicle_id, a.alias_id,
         CASE WHEN a.alias_id IS NULL THEN NULL
              WHEN d.development_vehicle_code = a.local_vehicle_code THEN concat('EXACT:', a.mapping_method)
              ELSE concat('NORMALIZED:', a.mapping_method) END AS method
  FROM development_plan d
  LEFT JOIN v_alias_approved a
    ON a.business_area     = 'DEVELOPMENT'
   AND a.norm_vehicle_code = norm_code(d.development_vehicle_code)
   AND d.plan_month BETWEEN a.valid_from AND a.valid_to
)
SELECT
  plan_record_id, development_vehicle_code, plan_year, plan_month, plan_version,
  planned_volume, planned_material_cost, planned_sales,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 1 THEN max(canonical_vehicle_id) END AS canonical_vehicle_id,
  CASE WHEN count(DISTINCT canonical_vehicle_id) = 0 THEN 'UNRESOLVED'
       WHEN count(DISTINCT canonical_vehicle_id) > 1 THEN 'CONFLICT'
       ELSE 'RESOLVED' END                                   AS resolution_status,
  nullif(concat_ws(',', array_sort(collect_set(method))), '')   AS resolution_method,
  nullif(concat_ws(',', array_sort(collect_set(alias_id))), '') AS alias_ids
FROM matched
GROUP BY plan_record_id, development_vehicle_code, plan_year, plan_month, plan_version,
         planned_volume, planned_material_cost, planned_sales;
""")

### 変換結果の内訳（どのルールで解決したか）

> **💡 解説**
> - **なぜ**：どのルール（完全一致・正規化・過去の履歴）で何行を救えたかを確かめます。
> - **利点**：対応表の改善が、どの工程の手作業を減らしたかを説明する材料になります。

In [ ]:
run_sql(r"""
SELECT 'SALES' AS area, resolution_status, resolution_method, count(*) AS rows, sum(sales_volume) AS volume
FROM v_sales_resolved GROUP BY ALL
UNION ALL
SELECT 'PRODUCTION', resolution_status, resolution_method, count(*), sum(production_volume)
FROM v_production_resolved GROUP BY ALL
UNION ALL
SELECT 'DEVELOPMENT', resolution_status, resolution_method, count(*), sum(planned_volume)
FROM v_plan_resolved GROUP BY ALL
ORDER BY area, resolution_status, resolution_method;
""")

## 2-7. 車種別採算分析用 View（共通機種ID × 月で計画・販売・生産を横断）

注意: 粒度を揃えてから結合する（各系を 共通機種ID × 月 に集約してから JOIN）。  
集約前に JOIN すると行が掛け算で増え、金額が重複計上される。  

> **💡 解説**
> - **なぜ**：計画・販売・生産を「共通機種ID × 月」でそろえて並べ、採算を見られるようにします。
> - **仕組み**：3つの系統を、それぞれ「共通機種ID × 月」に集計してから結合します（明細のまま結合すると行が掛け算で増え、金額が重複するため）。すべての組み合わせ（`keys`）を `UNION` で作ってから `LEFT JOIN` するので、どれかの系統にしか無い月も落ちません。
> - **利点**：1台あたりの材料費など、系統をまたいだ指標を正しく計算できます。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_vehicle_monthly_profitability
COMMENT '車種別採算分析用 View。共通機種ID × 月で計画・販売・生産を横断集計（金額: 百万円・架空）。RESOLVED のレコードのみ集計。'
AS
WITH plan AS (
  SELECT canonical_vehicle_id, plan_month AS month,
         sum(planned_volume) AS planned_volume,
         sum(planned_material_cost) AS planned_material_cost,
         sum(planned_sales) AS planned_sales
  FROM v_plan_resolved WHERE resolution_status = 'RESOLVED' GROUP BY ALL
), sales AS (
  SELECT canonical_vehicle_id, sales_month AS month,
         sum(sales_volume) AS sales_volume,
         sum(actual_sales) AS actual_sales
  FROM v_sales_resolved WHERE resolution_status = 'RESOLVED' GROUP BY ALL
), prod AS (
  SELECT canonical_vehicle_id, production_month AS month,
         sum(production_volume) AS production_volume,
         sum(actual_material_cost) AS actual_material_cost
  FROM v_production_resolved WHERE resolution_status = 'RESOLVED' GROUP BY ALL
), keys AS (
  SELECT canonical_vehicle_id, month FROM plan
  UNION SELECT canonical_vehicle_id, month FROM sales
  UNION SELECT canonical_vehicle_id, month FROM prod
)
SELECT
  k.canonical_vehicle_id,
  v.global_vehicle_name,
  k.month,
  plan.planned_volume,
  sales.sales_volume,
  prod.production_volume,
  plan.planned_sales,
  sales.actual_sales,
  sales.actual_sales - plan.planned_sales                           AS sales_variance,
  plan.planned_material_cost,
  prod.actual_material_cost,
  prod.actual_material_cost - plan.planned_material_cost            AS material_cost_variance,
  round(plan.planned_material_cost / nullif(plan.planned_volume, 0), 3)      AS planned_cost_per_unit,
  round(prod.actual_material_cost  / nullif(prod.production_volume, 0), 3)   AS actual_cost_per_unit
FROM keys k
LEFT JOIN plan  USING (canonical_vehicle_id, month)
LEFT JOIN sales USING (canonical_vehicle_id, month)
LEFT JOIN prod  USING (canonical_vehicle_id, month)
LEFT JOIN canonical_vehicle v ON v.canonical_vehicle_id = k.canonical_vehicle_id;
""")

### ドメインタグ（Domains UI が使えない場合の代替・検索性向上）

> **💡 解説**
> - **なぜ**：View にもドメインのタグを付け、業務のまとまりとして検索できるようにします。

In [ ]:
run_sql(r"""
ALTER VIEW v_vehicle_monthly_profitability SET TAGS ('handson_business_domain' = 'Vehicle Profitability', 'data_role' = 'analytics');
""")

## 2-8. 結果確認：Model Alpha 11th Gen（VEHICLE-001）の月次採算

> **💡 解説**
> - **なぜ**：結果を期待値と照らし合わせます。8月の材料費が突出していることも確認します。

In [ ]:
run_sql(r"""
SELECT month, planned_volume, sales_volume, production_volume,
       planned_sales, actual_sales, sales_variance,
       planned_material_cost, actual_material_cost, material_cost_variance,
       planned_cost_per_unit, actual_cost_per_unit
FROM v_vehicle_monthly_profitability
WHERE canonical_vehicle_id = 'VEHICLE-001'
ORDER BY month;
-- 期待値（抜粋）: 2025-08 実績材料費 7,150 / 計画 6,660 / 差額 +490（最大）
--               6か月合計 計画材料費 39,060 / 実績材料費 38,730 / 差額 -330
""")

# 2-9. 集計から除外されたレコード（次の Exercise 5 で扱う）

> **💡 解説**
> - **なぜ**：集計から外したデータを隠さず、一覧にします。どれだけの金額が集計に入っていないかを、後で経営者にも示せるようにするためです。
> - **仕組み**：3つの変換 View から、RESOLVED 以外の行を `UNION ALL` でまとめます。

In [ ]:
run_sql(r"""
CREATE OR REPLACE VIEW v_unresolved_records
COMMENT '共通機種IDに変換できず集計から除外されたレコード（UNRESOLVED / CONFLICT）'
AS
SELECT 'SALES' AS business_area, sales_record_id AS record_id, country, sales_vehicle_code AS source_code,
       sales_month AS month, sales_volume AS volume, actual_sales AS amount, 'actual_sales' AS amount_type, resolution_status
FROM v_sales_resolved WHERE resolution_status <> 'RESOLVED'
UNION ALL
SELECT 'PRODUCTION', production_record_id, 'JP', mto_code,
       production_month, production_volume, actual_material_cost, 'actual_material_cost', resolution_status
FROM v_production_resolved WHERE resolution_status <> 'RESOLVED'
UNION ALL
SELECT 'DEVELOPMENT', plan_record_id, 'GLOBAL', development_vehicle_code,
       plan_month, planned_volume, planned_material_cost, 'planned_material_cost', resolution_status
FROM v_plan_resolved WHERE resolution_status <> 'RESOLVED';
""")

> **💡 解説**
> - **なぜ**：除外されたレコードを確認します。Exercise 5 で、これを分類します。

In [ ]:
run_sql(r"""
SELECT * FROM v_unresolved_records ORDER BY business_area, record_id;
-- 期待値: 販売 4行（CN-X12 / JP-A11-SE / TH-Z999 / KR-Q777）、生産 2行（MTO-988 UNRESOLVED / MTO-990 CONFLICT）
""")